In [ ]:
from typing import TypedDict, List, Optional
from langgraph.graph import StateGraph, END

In [ ]:
class AgentState(TypedDict):
    tasks: List[dict]
    current: Optional[dict]
    results: List[dict]
    step: int

In [ ]:
def supervisor(state: AgentState) -> AgentState:
    state["step"] += 1
    if state["tasks"]:
        state["current"] = state["tasks"].pop(0)
        print(f"[step {state['step']}] supervisor -> route {state['current']['kind']:<5} {state['current']['payload']}")
    else:
        state["current"] = None
        print(f"[step {state['step']}] supervisor -> queue empty, finish")
    return state

In [ ]:
def math_worker(state: AgentState) -> AgentState:
    payload = state["current"]["payload"]
    out = sum(payload)
    state["results"].append({"kind": "math", "in": payload, "out": out})
    print(f"            math       -> sum({payload}) = {out}")
    return state

In [ ]:
def text_worker(state: AgentState) -> AgentState:
    payload = state["current"]["payload"]
    out = payload.upper()
    state["results"].append({"kind": "text", "in": payload, "out": out})
    print(f"            text       -> {payload!r} -> {out!r}")
    return state

In [ ]:
def log_worker(state: AgentState) -> AgentState:
    payload = state["current"]["payload"]
    state["results"].append({"kind": "log", "in": payload, "out": payload})
    print(f"            log        -> {payload!r}")
    return state

In [ ]:
def route(state: AgentState) -> str:
    if state["current"] is None:
        return "done"
    return state["current"]["kind"]

In [ ]:
graph = StateGraph(AgentState)

In [ ]:
graph.add_node("supervisor", supervisor)
graph.add_node("math", math_worker)
graph.add_node("text", text_worker)
graph.add_node("log", log_worker)

graph.set_entry_point("supervisor")

graph.add_conditional_edges(
    "supervisor",
    route,
    {"math": "math", "text": "text", "log": "log", "done": END},
)

graph.add_edge("math", "supervisor")
graph.add_edge("text", "supervisor")
graph.add_edge("log", "supervisor")

In [ ]:
test = graph.compile()

In [ ]:
from IPython.display import Image, display
display(Image(test.get_graph().draw_mermaid_png()))

In [ ]:
result = test.invoke({
    "tasks": [
        {"kind": "math", "payload": [1, 2, 3, 4]},
        {"kind": "text", "payload": "hello world"},
        {"kind": "log",  "payload": "system online"},
        {"kind": "math", "payload": [10, 20]},
    ],
    "current": None,
    "results": [],
    "step": 0,
})

In [ ]:
result